## 1. Setup

In [ ]:
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path("../../src").resolve()))
SMOKE = os.environ.get("CONDPRED_SMOKE") == "1"
from condpred.experiments.e10 import build_initial_spec, run_leakage_e10, load_market, select_pairs, run_pipeline, E10_MODULES
from condpred import report
SPEC = build_initial_spec(SMOKE)
out_dir = Path("../tables/condpred/E10") / ("smoke" if SMOKE else "")
fig_dir = out_dir / "figures"
out_dir.mkdir(parents=True, exist_ok=True); fig_dir.mkdir(parents=True, exist_ok=True)
print("SMOKE", SMOKE, SPEC["step"], SPEC["first_test_year"])

## 2. Write spec

In [ ]:
report.write_spec(SPEC, out_dir, modules=E10_MODULES, completion_keys=frozenset({"pairs", "snapshot_file"}))
print("spec written", out_dir / "spec.json")

## 3. Leakage test

In [ ]:
checks = run_leakage_e10()
print(checks)
assert all(ok for _, ok in checks)

## 4. Snapshot, closing-time check, pair list

In [ ]:
market = load_market(SPEC)
pairs, pair_report = select_pairs(market["returns"], SMOKE)
print(f"{len(pairs)} pairs kept")
print(pair_report.to_string())

## 5. Fixed models and targets

In [ ]:
print(SPEC["fixed_models"]); print(SPEC["targets"])

## 6. Features

In [ ]:
print(SPEC["features"])

## 7. Walk-forward

In [ ]:
result = run_pipeline(SPEC, out_dir, fig_dir)
print(result["primary"])

## 8. Decision

In [ ]:
import pandas as pd
print(pd.read_csv(out_dir / "decision.csv").to_string())

## 9. Secondary

In [ ]:
sec = pd.read_csv(out_dir / "secondary.csv") if (out_dir / "secondary.csv").exists() else pd.DataFrame()
print(sec.to_string() if len(sec) else "no secondary rows")

## 10. Figures and tables

In [ ]:
print(sorted(p.name for p in out_dir.iterdir()))
print(sorted(p.name for p in fig_dir.iterdir()))